In [9]:
import numpy as np


In [10]:
import yfinance as yf
import pandas as pd


class YahooFinancePipeline:



    @staticmethod
    def get_stock_info(ticker):

        stock = yf.Ticker(ticker)

        info = stock.info

        return {

            "ticker": ticker,

            "company_name":
                info.get("longName"),

            "sector":
                info.get("sector"),

            "industry":
                info.get("industry"),

            "beta":
                info.get("beta"),

            "market_cap":
                info.get("marketCap")

        }



    @staticmethod
    def get_price_history(

        ticker,

        start_date,

        end_date

    ):

        df = yf.download(

            ticker,

            start=start_date,

            end=end_date,

            progress=False

        )

        df.reset_index(inplace=True)

        return df



    @staticmethod
    def get_benchmark_data(

        start_date,

        end_date

    ):

        benchmark = yf.download(

            "^GSPC",

            start=start_date,

            end=end_date,

            progress=False

        )

        benchmark.reset_index(inplace=True)

        return benchmark


    @staticmethod
    def calculate_returns(df):

        df = df.copy()

        df["Daily_Return"] = (

            df["Close"]

            .pct_change()

        )

        df.dropna(inplace=True)

        return df



    @staticmethod
    def clean_data(df):

        df = df.copy()

        df.dropna(inplace=True)

        df.drop_duplicates(inplace=True)

        return df



    @classmethod
    def build_dataset(

        cls,

        ticker,

        start_date,

        end_date

    ):



        stock_info = cls.get_stock_info(
            ticker
        )



        history = cls.get_price_history(

            ticker,

            start_date,

            end_date

        )

        history = cls.clean_data(
            history
        )

        history = cls.calculate_returns(
            history
        )



        benchmark = cls.get_benchmark_data(

            start_date,

            end_date

        )

        benchmark = cls.clean_data(
            benchmark
        )

        benchmark = cls.calculate_returns(
            benchmark
        )

        return {

            "metadata": stock_info,

            "stock_history": history,

            "benchmark_history": benchmark

        }


In [11]:
pipeline = YahooFinancePipeline()

dataset = pipeline.build_dataset(

    ticker="NVDA",

    start_date="2024-01-01",

    end_date="2025-01-01"

)
print("metadata")
print(dataset["metadata"])

print("stock Hitory")

print(
    dataset["stock_history"].head()
)

print("benchmark history")

print(
    dataset["benchmark_history"].head()
)

metadata
{'ticker': 'NVDA', 'company_name': 'NVIDIA Corporation', 'sector': 'Technology', 'industry': 'Semiconductors', 'beta': 2.244, 'market_cap': 5201459675136}
stock Hitory
Price        Date      Close       High        Low       Open     Volume  \
Ticker                  NVDA       NVDA       NVDA       NVDA       NVDA   
1      2024-01-03  47.539940  48.154562  47.291091  47.455992  320896000   
2      2024-01-04  47.968685  48.470377  47.478983  47.737823  306535000   
3      2024-01-05  49.067017  49.516743  48.276499  48.432406  415039000   
4      2024-01-08  52.221085  52.243074  49.448781  49.481761  642510000   
5      2024-01-09  53.107540  54.291818  51.658425  52.368993  773100000   

Price  Daily_Return  
Ticker               
1         -0.012436  
2          0.009019  
3          0.022897  
4          0.064281  
5          0.016975  
benchmark history
Price        Date        Close         High          Low         Open  \
Ticker                   ^GSPC        ^GSPC  

/tmp/ipykernel_12955/1988333961.py:50: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
/tmp/ipykernel_12955/1988333961.py:77: FutureWarning: YF.download() has changed argument auto_adjust default to True
  benchmark = yf.download(


In [12]:
dataset = YahooFinancePipeline.build_dataset(
    "NVDA",
    "2024-01-01",
    "2025-01-01"
)

stock_returns = dataset[
    "stock_history"
]["Daily_Return"]

benchmark_returns = dataset[
    "benchmark_history"
]["Daily_Return"]

/tmp/ipykernel_12955/1988333961.py:50: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
/tmp/ipykernel_12955/1988333961.py:77: FutureWarning: YF.download() has changed argument auto_adjust default to True
  benchmark = yf.download(


In [13]:
class SharpeRatio:

    @staticmethod
    def calculate(returns):

        excess_returns = returns.dropna()

        sharpe = (
            np.mean(excess_returns)
            /
            np.std(excess_returns)
        ) * np.sqrt(252)

        return round(sharpe, 4)

In [14]:
nvda = yf.download(
    "NVDA",
    start="2024-01-01",
    end="2025-01-01"
)

nvda["Daily_Return"] = nvda["Close"].pct_change()

stock_returns = nvda["Daily_Return"].dropna()

/tmp/ipykernel_12955/3550297209.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nvda = yf.download(
[*********************100%***********************]  1 of 1 completed


In [15]:
SharpeRatio.calculate(stock_returns)

np.float64(2.2282)

In [16]:
print("SharpeRatio" in globals())
print("stock_returns" in globals())
print("dataset" in globals())

True
True
True
